# Efficient Full Multilevel MOT: Single Trajectory

This is the full cooling-plus-repumper counterpart of the **Single Trajectory** cell in `notebooks/mot_simple/disc_sampling_geometry.ipynb`. It uses the efficient adiabatic-elimination population rate equations from `docs/mot_multilevel/EFFICIENT_MOT.md`: at each fixed external timestep it solves the quasi-steady hyperfine/Zeeman populations, obtains mean force and recoil diffusion, and advances the atom with a Langevin step. Individual photon jumps are not tracked here.

In [ ]:
%matplotlib widget

from dataclasses import replace
from datetime import datetime
from pathlib import Path
from time import perf_counter

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pmot.magnetic_fields import default_anti_helmholtz_config
from pmot.magnetic_field_plotting import plot_magnetic_component_grid
from pmot.launch_geometry import build_incident_disc_from_angles
from pmot.mot_multilevel import build_multilevel_mot_beams, default_multilevel_mot_config
from pmot.mot_multilevel.rate_equations import (
    RateEquationAtomState, RateEquationTrajectoryConfig,
    build_rate_equation_model, simulate_rate_equation_trajectory,
)
from pmot.mot_multilevel.rate_diagnostics import (
    plot_hyperfine_manifold_occupation,
    plot_rate_equation_force_potential_curves,
    plot_rate_equation_time_diagnostics,
    save_rate_equation_trajectory, summarize_rate_equation_trajectory,
)
from pmot.mot_multilevel.screening import draw_multilevel_mot_beam_volumes

COIL_CONFIG = default_anti_helmholtz_config()
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_DIRECTORY = PROJECT_ROOT / 'outputs' / 'trajectories' / 'mot_multilevel' / 'notebook_rate_equation'

In [ ]:
def build_launch_point(theta_deg, phi_deg, radial_distance_mm, theta_prime_deg, impact_parameter_mm):
    disc = build_incident_disc_from_angles(
        disc_index=0, radial_distance_m=1e-3 * radial_distance_mm,
        theta_rad=np.deg2rad(theta_deg), phi_rad=np.deg2rad(phi_deg),
    )
    theta_prime = np.deg2rad(theta_prime_deg)
    offset = 1e-3 * impact_parameter_mm * (
        np.cos(theta_prime) * np.asarray(disc.basis_u)
        + np.sin(theta_prime) * np.asarray(disc.basis_v)
    )
    position = np.asarray(disc.center_position_m) + offset
    return tuple(position), tuple(disc.incident_unit_vector)


def status_style(record, summary):
    if record.termination_reason == 'escaped':
        return 'ESCAPED', '#b91c1c'
    if summary['bounded_trapping_candidate']:
        return 'BOUNDED TRAPPING CANDIDATE', '#15803d'
    if summary['two_core_entry_candidate']:
        return 'TWO-CORE-ENTRY CANDIDATE', '#0f766e'
    return 'DURATION COMPLETE — NOT YET BOUNDED', '#1d4ed8'

## Single Trajectory

The controls match the simplified notebook, including physical duration `T` and fixed external timestep `dt`. Repumping and Langevin recoil diffusion are enabled by default. The population solve includes all dipole-allowed repump channels, including (F=1\rightarrow F'=0).

Each run produces the 3D trajectory; Cartesian position, velocity, per-beam scattering rate, and optical-force histories; steady-state restoring/damping force curves; effective one-dimensional potential curves; magnetic-field component surfaces; and a hyperfine-manifold occupation histogram. Because this is a population-rate model, a histogram bin is (100/T) times the time integral of that manifold's probability, summed over all of its (m_F) substates. It is not a count of discrete state jumps.

In [ ]:
single_theta = widgets.FloatSlider(description='theta [deg]', min=0, max=90, step=1, value=35, continuous_update=False)
single_phi = widgets.FloatSlider(description='phi [deg]', min=0, max=90, step=1, value=20, continuous_update=False)
single_d = widgets.FloatSlider(description='d [mm]', min=5, max=30, step=.5, value=15, continuous_update=False)
single_theta_prime = widgets.FloatSlider(description="theta' [deg]", min=0, max=360, step=1, value=0, continuous_update=False)
single_s = widgets.FloatSlider(description='s [mm]', min=0, max=12, step=.25, value=3, continuous_update=False)
single_v0 = widgets.FloatSlider(description='v0 [m/s]', min=0, max=30, step=.5, value=5, continuous_update=False)
single_duration = widgets.FloatSlider(description='T [ms]', min=1, max=80, step=1, value=25, continuous_update=False)
single_dt = widgets.FloatSlider(description='dt [us]', min=1, max=20, step=1, value=5, continuous_update=False)
single_seed = widgets.BoundedIntText(description='seed', min=0, max=2_000_000_000, value=20260819)
single_repumper = widgets.Checkbox(description='repumper enabled', value=True, indent=False)
single_diffusion = widgets.Checkbox(description='recoil diffusion', value=True, indent=False)
single_save = widgets.Checkbox(description='save CSV/NPZ', value=True, indent=False)
single_button = widgets.Button(description='Run Efficient Full MOT', button_style='primary')
single_output = widgets.Output()

def run_single_trajectory(_=None):
    with single_output:
        single_output.clear_output(wait=True)
        plt.close('all')
        position, direction = build_launch_point(
            single_theta.value, single_phi.value, single_d.value,
            single_theta_prime.value, single_s.value,
        )
        velocity = tuple(single_v0.value * np.asarray(direction))
        config = replace(default_multilevel_mot_config(), repumper_enabled=single_repumper.value)
        model = build_rate_equation_model(config.natural_linewidth_rad_per_s)
        beams = build_multilevel_mot_beams(config=config)
        numerical = RateEquationTrajectoryConfig(
            time_step_s=1e-6 * single_dt.value, include_diffusion=single_diffusion.value,
            seed=single_seed.value, escape_radius_m=30e-3,
        )
        print(f'Running T={single_duration.value:.1f} ms, dt={single_dt.value:.1f} us, steps={int(np.ceil(1000*single_duration.value/single_dt.value)):,} ...', flush=True)
        start = perf_counter()
        record = simulate_rate_equation_trajectory(
            RateEquationAtomState(position, velocity), 1e-3 * single_duration.value, COIL_CONFIG,
            beams=beams, model=model, config=config, trajectory_config=numerical,
        )
        wall_time = perf_counter() - start
        summary = summarize_rate_equation_trajectory(record)
        status, status_color = status_style(record, summary)
        stem = None
        if single_save.value:
            stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            stem = OUTPUT_DIRECTORY / f'trajectory_{stamp}_seed_{single_seed.value}'
        def figure_path(suffix):
            return stem.with_name(f'{stem.name}_{suffix}.png') if stem is not None else None
        positions_mm = 1e3 * np.asarray(record.positions_m)
        figure = plt.figure(figsize=(9, 7.5), constrained_layout=True)
        figure.patch.set_facecolor('#fbfaf6')
        axis = figure.add_subplot(111, projection='3d')
        axis.set_facecolor('#fbfaf6')
        draw_multilevel_mot_beam_volumes(axis, beams)
        axis.plot(*positions_mm.T, color='#0f766e', linewidth=2.2)
        axis.scatter(*positions_mm[0], color='#b91c1c', s=50, label='start')
        axis.scatter(*positions_mm[-1], color='#111827', s=50, label='end')
        axis.scatter(0, 0, 0, color='#7c3aed', s=36, label='trap center')
        axis.text2D(.03, .95, status, transform=axis.transAxes, color=status_color, fontsize=12, fontweight='bold')
        axis.set_title(
            'Efficient Full Multilevel MOT — Single Launch Trajectory\n'
            f"d={single_d.value:.1f} mm, theta={single_theta.value:.1f} deg, phi={single_phi.value:.1f} deg, theta'={single_theta_prime.value:.1f} deg, "
            f's={single_s.value:.1f} mm, v0={single_v0.value:.1f} m/s, T={single_duration.value:.1f} ms, dt={single_dt.value:.1f} us'
        )
        axis.set(xlabel='x [mm]', ylabel='y [mm]', zlabel='z [mm]', xlim=(-32,32), ylim=(-32,32), zlim=(-32,32))
        axis.set_box_aspect((1,1,1)); axis.legend(loc='best')
        if figure_path('3d') is not None:
            figure_path('3d').parent.mkdir(parents=True, exist_ok=True)
            figure.savefig(figure_path('3d'), dpi=180, bbox_inches='tight')

        plot_rate_equation_time_diagnostics(record, beams, figure_path('time_diagnostics'))
        plot_hyperfine_manifold_occupation(record, model, figure_path('hyperfine_occupation'))
        print('Computing quasi-steady force and effective-potential curves ...', flush=True)
        plot_rate_equation_force_potential_curves(
            model, beams, COIL_CONFIG, config, figure_path('force_potential_curves'),
            position_extent_m=8e-3, velocity_extent_m_per_s=15.0, sample_count=81,
        )
        plot_magnetic_component_grid(
            COIL_CONFIG, extent_m=10e-3, samples_per_axis=61,
            path=figure_path('magnetic_field_surfaces'),
        )
        plt.show()
        print(f"termination={record.termination_reason}; wall time={wall_time:.3f} s; saved steps={len(record.times_s)-1:,}")
        print(f"core entries={summary['core_entry_count']}; min/final radius={1e3*summary['minimum_radius_m']:.4g}/{1e3*summary['final_radius_m']:.4g} mm")
        print(f"final-window maximum radius={1e3*summary['final_window_maximum_radius_m']:.4g} mm; bounded candidate={summary['bounded_trapping_candidate']}")
        if single_save.value:
            paths = save_rate_equation_trajectory(record, model, beams, stem, metadata={
                'duration_ms': single_duration.value, 'time_step_us': single_dt.value,
                'seed': single_seed.value, 'repumper_enabled': single_repumper.value,
                'diffusion_enabled': single_diffusion.value, 'summary': summary,
            })
            paths.extend([
                figure_path('3d'), figure_path('time_diagnostics'),
                figure_path('hyperfine_occupation'), figure_path('force_potential_curves'),
                figure_path('magnetic_field_surfaces'),
            ])
            print('Saved data and figures:'); [print(f'  {path}') for path in paths]

single_button.on_click(run_single_trajectory)
display(widgets.VBox([
    widgets.HBox([single_theta, single_phi, single_d]),
    widgets.HBox([single_theta_prime, single_s, single_v0]),
    widgets.HBox([single_duration, single_dt, single_seed]),
    widgets.HBox([single_repumper, single_diffusion, single_save, single_button]),
    single_output,
]))

## Model boundary

This population-only approximation includes Doppler/Zeeman shifts, local polarization, optical pumping, repumping, mean force, gravity, and recoil diffusion. It intentionally excludes optical coherences, coherent dark states, and sub-Doppler polarization-gradient cooling. The expensive hyperfine jump animation remains available separately through `pmot.mot_multilevel.jump_visualization`; it is not used by this trajectory solver.